In [20]:
from stable_baselines3.sac import SAC
from stable_baselines3.sac.policies import MultiInputPolicy
from imitation.data.types import DictObs
import numpy as np
import pandas as pd

import gymnasium as gym
from flycraft.env import FlyCraftEnv

import sys
from pathlib import Path
from time import time
import logging
from tqdm import tqdm
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT_DIR = NOTEBOOK_DIR.parent.parent.parent 
# PROJECT_ROOT_DIR = Path(__file__).absolute().parent.parent.parent.parent
print(PROJECT_ROOT_DIR)
if str(PROJECT_ROOT_DIR.absolute()) not in sys.path:
    sys.path.append(str(PROJECT_ROOT_DIR.absolute()))

from utils_my.sb3.vec_env_helper import make_env
from utils_my.sb3.my_wrappers import ScaledObservationWrapper, ScaledActionWrapper


def load_random_transitions_from_csv_files(
        data_dir: Path, 
        trajectory_save_prefix: str="traj",
        env_config_file: Path=PROJECT_ROOT_DIR / "configs" / "env" / "env_config_for_sac.json",
        n_env = 1
    ):

    start_time = time()
    res_file = data_dir / "res.csv"
    res_df = pd.read_csv(res_file)

    valid_lengths = []
    traj_cnt = 0
    mycount = 0

    origin_env = FlyCraftEnv(config_file=env_config_file)
    
    for index, row in tqdm(res_df.iterrows(), total=res_df.shape[0]):
        target_v, target_mu, target_chi, cur_length = row["v"], row["mu"], row["chi"], row["length"]

        # 过滤掉desired goal范围之外的专家轨迹
        if not (
            (origin_env.env_config["goal"]["v_min"] <= target_v <= origin_env.env_config["goal"]["v_max"])
            and
            (origin_env.env_config["goal"]["mu_min"] <= target_mu <= origin_env.env_config["goal"]["mu_max"])
            and
            (origin_env.env_config["goal"]["chi_min"] <= target_chi <= origin_env.env_config["goal"]["chi_max"])
        ):
            mycount += 1
            continue

        if cur_length > 0:
            valid_lengths.append(cur_length-1)
            traj_cnt += 1

    # 计算统计量
    if valid_lengths:
        avg_length = np.mean(valid_lengths)
        std_length = np.std(valid_lengths)
        min_length = np.min(valid_lengths)
        max_length = np.max(valid_lengths)
        who_length = np.sum(valid_lengths)
        stats_message = (
            f"轨迹长度统计:\n"
            f" - 平均长度: {avg_length:.2f} ± {std_length:.2f}\n"
            f" - 最小长度: {min_length}\n"
            f" - 最大长度: {max_length}\n"
            f" - 总轨迹数: {traj_cnt}\n"
            f" - 过滤掉的轨迹数: {mycount}"
            f"总transit数：{who_length}"
        )
        
        print(stats_message)




/home/sen/pythonprojects/fly-craft-examples


# aug 4 

In [21]:
load_random_transitions_from_csv_files(data_dir=Path("/home/sen/pythonprojects/fly-craft-examples/demonstrations/data/10hz_10_5_5_iter_4_aug"),trajectory_save_prefix="my_f16trace",env_config_file=PROJECT_ROOT_DIR/"configs/env/D2D/env_config_for_sac_medium_b_2.json")

load config from: /home/sen/pythonprojects/fly-craft-examples/configs/env/D2D/env_config_for_sac_medium_b_2.json


100%|██████████| 50715/50715 [00:01<00:00, 42103.48it/s]

轨迹长度统计:
 - 平均长度: 62.14 ± 23.01
 - 最小长度: 8.0
 - 最大长度: 322.0
 - 总轨迹数: 3575
 - 过滤掉的轨迹数: 47140总transit数：222146.0


In [22]:
load_random_transitions_from_csv_files(data_dir=Path("/home/sen/pythonprojects/fly-craft-examples/demonstrations/data/10hz_10_5_5_iter_1_aug"),trajectory_save_prefix="traj",env_config_file=PROJECT_ROOT_DIR/"configs/env/D2D/env_config_for_sac_medium_b_2.json")

load config from: /home/sen/pythonprojects/fly-craft-examples/configs/env/D2D/env_config_for_sac_medium_b_2.json


100%|██████████| 50715/50715 [00:01<00:00, 42133.37it/s]


轨迹长度统计:
 - 平均长度: 163.40 ± 129.17
 - 最小长度: 9.0
 - 最大长度: 1182.0
 - 总轨迹数: 1909
 - 过滤掉的轨迹数: 47140总transit数：311930.0
